---

# Veyra: Cattle Muzzle Identification

*Training and open-set evaluation of the muzzle model that tells Veyra which registered animal a capture belongs to, or that it belongs to none of them.*

---

## Team Veyra

- Member 1 (Vision and data)
- Member 2
- Member 3
- Member 4

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Profiling**](#3)
4. [**Animal-Level Splits**](#4)
5. [**Evaluation Protocol**](#5)
6. [**Baseline: SIFT and RANSAC**](#6)
7. [**Zero-Shot Foundation Models**](#7)
8. [**DINOv3 and ArcFace Fine-Tuning**](#8)
9. [**Hybrid: Embedding Retrieval and SIFT Verification**](#9)
10. [**Results Comparison**](#10)
11. [**Export to ONNX**](#11)


---

# Introduction <a name="1"></a>

---

## Overview

*Veyra turns smallholder cattle into evidence records that a bank can review as potential collateral. The first question every record must answer is which animal it describes, and the cattle muzzle carries a ridge-and-bead pattern that is distinctive for each individual.*

*The task is individual identification rather than species classification. A new capture is compared against a gallery of registered animals, and the system must either name a candidate for a human verifier or report that no registered animal matches.*

## Aim

*Build a muzzle retrieval model that ranks the correct registered animal first and keeps the false-match rate on unregistered animals at or below 5%, then compare it against a classical SIFT baseline and zero-shot DINOv2 and DINOv3 baselines on the same held-out animals.*

## Metric

***Correct match at the operating threshold** is the share of queries from registered animals whose top candidate is correct and scores above the threshold. The threshold is the score that allows at most 5% of validation queries from unknown animals to match anything. It is appropriate here because Veyra must reject animals that are not in the registry, so closed-set accuracy alone would overstate performance.*

$$\text{Correct match} = \frac{|\{q \in Q_{reg} : \hat{a}(q) = a(q) \wedge s(q) > \tau\}|}{|Q_{reg}|}\qquad \text{FMR}_{unk} = \frac{|\{q \in Q_{unk} : s(q) > \tau\}|}{|Q_{unk}|}$$

*Rank-1 and top-5 retrieval rates are reported alongside as closed-set diagnostics.*

## Dataset

*Data sourced from Zenodo, Beef Cattle Muzzle/Noseprint database (Li, Erickson, and Xiong, 2022, CC BY 4.0) at: https://zenodo.org/records/6324361*

*The dataset contains 4,923 cropped muzzle images of 268 US feedlot cattle, with one folder per animal and roughly 18 images per animal on average.*

*The dataset is used to train the embedding model on one group of animals and to measure retrieval and unknown-animal rejection on animals the model has never seen.*

```
li2022_muzzle/
└── <image_root>/
     ├── <animal_id_1>/
     |    ├── <image>.jpg
     |    └── ...
     └── <animal_id_268>/
```

## Approach: DINOv3 with ArcFace Metric Learning

*DINOv3 is a self-supervised Vision Transformer distilled from a 7B-parameter teacher trained on 1.7 billion images. Its patch-level features stay sharp, which suits the fine ridge texture of a muzzle. DINOv2 is evaluated alongside it in the zero-shot stage, so the backbone choice rests on measured results. ArcFace adds an angular margin between identities during training, which pulls images of the same animal together on a hypersphere and pushes different animals apart.*

*At inference time the classification head is discarded. Each image becomes a normalised embedding, and identification is a cosine-similarity search against the gallery, so a new animal is registered by adding its embedding without retraining.*

```
Muzzle crop [224 x 224]
    |
    +-- DINOv3 ViT-S/16 backbone (last blocks fine-tuned) [384]
         |
    Projection + L2 normalisation [512]
         |
    Training: ArcFace head over training animals
    Inference: cosine similarity against the gallery -> ranked candidates
```

*Animals, never images, are assigned to train, validation, and test groups. Validation and test each hold registered animals and unknown animals that never enter the gallery, following the evaluation protocol in Section 4.3 of the Veyra PRD.*

---

# Initialization <a name="2"></a>

---

## Environment Setup

To reproduce results without re-running training, assign `Settings.RUN_NAME` to `'INFERENCE'` and then run all cells. The notebook targets Kaggle with the GPU T4 x2 accelerator and also runs locally on Apple Silicon (MPS). On Kaggle, every attached dataset under `/kaggle/input` is searched for animal folders.

In [ ]:
!nvidia-smi || echo 'No NVIDIA GPU detected'

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q numpy pandas matplotlib pillow tqdm tabulate opencv-contrib-python-headless torch torchvision timm onnx onnxruntime

## Import Libraries

In [ ]:
import json
import math
import os
import random
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from tqdm.auto import tqdm

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

## Settings

All paths, hyperparameters, and environment flags are centralised here. Switching between Kaggle and local execution requires no changes outside this class. `IMAGE_ROOT` is the folder whose subfolders are individual animals.

In [ ]:
class Settings:
    SEED       = 42
    RUN_NAME   = 'dinov3_arcface_v1'
    _ON_KAGGLE = Path('/kaggle/input').exists()

    DATA_DIR   = (
        Path('/kaggle/input')
        if _ON_KAGGLE else Path('../data/raw/li2022_muzzle')
    )
    IMAGE_ROOT = None
    OUTPUT_DIR = Path('/kaggle/working') if _ON_KAGGLE else Path('../runs')
    RUN_DIR    = OUTPUT_DIR / RUN_NAME
    FIG_DIR    = RUN_DIR / 'figures'
    SPLIT_PATH = Path('/kaggle/working/muzzle_v1.json') if _ON_KAGGLE else Path('../splits/muzzle_v1.json')
    RESULT_DIR = Path('/kaggle/working') if _ON_KAGGLE else Path('../results')
    EXPORT_DIR = Path('/kaggle/working') if _ON_KAGGLE else Path('../../vision/models')

    TRAIN_FRACTION     = 0.50
    VAL_FRACTION       = 0.20
    UNKNOWN_FRACTION   = 0.25
    GALLERY_PER_ANIMAL = 1
    TARGET_UNKNOWN_FMR = 0.05

    SIFT_MAX_SIDE   = 480
    SIFT_FEATURES   = 800
    SIFT_RATIO      = 0.75
    SIFT_SHORTLIST  = 10
    SIFT_MIN_MATCH  = 8
    HYBRID_TOP_K    = 5

    BACKBONE        = 'vit_small_patch16_dinov3.lvd1689m'
    ZERO_SHOT_BACKBONES = ['vit_small_patch14_dinov2.lvd142m', 'vit_small_patch16_dinov3.lvd1689m']
    IMG_SIZE        = 224
    EMBED_DIM       = 512
    TRAINABLE_BLOCKS = 4
    ARC_SCALE       = 30.0
    ARC_MARGIN      = 0.30
    EPOCHS          = 20
    N_GPUS          = torch.cuda.device_count()
    BATCH_SIZE      = 32 * max(1, N_GPUS)
    LR_BACKBONE     = 2e-5
    LR_HEAD         = 1e-3
    WEIGHT_DECAY    = 0.05
    WARMUP_EPOCHS   = 1
    NUM_WORKERS     = 4 if _ON_KAGGLE else 0

    DEVICE = (
        'cuda' if torch.cuda.is_available()
        else 'mps' if torch.backends.mps.is_available() else 'cpu'
    )

CFG = Settings()
for d in (CFG.RUN_DIR, CFG.FIG_DIR, CFG.SPLIT_PATH.parent, CFG.RESULT_DIR, CFG.EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f'Device: {CFG.DEVICE}  GPUs: {CFG.N_GPUS}  Batch size: {CFG.BATCH_SIZE}  Run: {CFG.RUN_NAME}')

## Load Dataset

Every folder that directly contains images is treated as one animal, and its name becomes the animal ID. This works whether the archive was extracted with or without an extra top-level folder.

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

image_paths = sorted(p for p in CFG.DATA_DIR.rglob('*') if p.suffix.lower() in IMAGE_EXTS)
CFG.IMAGE_ROOT = Path(os.path.commonpath([p.parent.parent for p in image_paths]))

df = pd.DataFrame({
    'image':  [str(p.relative_to(CFG.IMAGE_ROOT)) for p in image_paths],
    'animal': [p.parent.name for p in image_paths],
})
animals = df.groupby('animal')['image'].apply(sorted).to_dict()
print(f'Image root: {CFG.IMAGE_ROOT}')
print(f'Images: {len(df)}  Animals: {len(animals)}')

---

# Data Profiling <a name="3"></a>

---

The number of images per animal decides how many query images remain after one image per registered animal goes into the gallery, and the image resolution decides how much muzzle detail survives resizing to 224 pixels. Both are checked before any split is made.

In [ ]:
counts = df.groupby('animal').size()
sizes = pd.DataFrame(
    [Image.open(CFG.IMAGE_ROOT / p).size for p in df['image']],
    columns=['width', 'height'],
)

print('Images per animal')
print(counts.describe().round(1).to_string())
print('\nImage size (pixels)')
print(sizes.describe().round(0).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(counts, bins=30, color='#4C72B0')
axes[0].set(title='Images per animal', xlabel='Images', ylabel='Animals')
axes[1].scatter(sizes['width'], sizes['height'], s=4, alpha=0.4, color='#4C72B0')
axes[1].set(title='Image size', xlabel='Width (px)', ylabel='Height (px)')
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'profile.png', dpi=120)
plt.show()

A grid of four images from each of four random animals shows how much the same muzzle varies across captures, which is the variation the model must tolerate.

In [ ]:
rng = random.Random(CFG.SEED)
sample_animals = rng.sample(sorted(animals), 4)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for r, a in enumerate(sample_animals):
    for c, rel in enumerate(rng.sample(animals[a], min(4, len(animals[a])))):
        axes[r, c].imshow(Image.open(CFG.IMAGE_ROOT / rel).convert('RGB'))
        axes[r, c].set_title(a if c == 0 else '', fontsize=9, loc='left')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'samples.png', dpi=120)
plt.show()

---

# Animal-Level Splits <a name="4"></a>

---

Animals are assigned to train, validation, and test groups so that no animal contributes images to more than one group. Within validation and test, a quarter of the animals are held out as unknown animals that never enter the gallery. The split is saved once and reloaded on later runs, so every method is scored on exactly the same animals.

In [ ]:
def split_group(ids: list[str], rng: random.Random) -> dict:
    ids = sorted(ids)
    rng.shuffle(ids)
    n_unknown = round(len(ids) * CFG.UNKNOWN_FRACTION)
    unknown, registered = ids[:n_unknown], ids[n_unknown:]
    gallery, queries = {}, []
    for a in registered:
        imgs = animals[a][:]
        rng.shuffle(imgs)
        gallery[a] = imgs[:CFG.GALLERY_PER_ANIMAL]
        queries += [{'image': i, 'animal': a, 'registered': True} for i in imgs[CFG.GALLERY_PER_ANIMAL:]]
    for a in unknown:
        queries += [{'image': i, 'animal': a, 'registered': False} for i in animals[a]]
    return {'registered': sorted(registered), 'unknown': sorted(unknown), 'gallery': gallery, 'queries': queries}


def make_split() -> dict:
    rng = random.Random(CFG.SEED)
    usable = sorted(a for a, imgs in animals.items() if len(imgs) > CFG.GALLERY_PER_ANIMAL)
    rng.shuffle(usable)
    n_train = round(len(usable) * CFG.TRAIN_FRACTION)
    n_val = round(len(usable) * CFG.VAL_FRACTION)
    return {
        'dataset': 'li2022_muzzle',
        'seed': CFG.SEED,
        'train': sorted(usable[:n_train]),
        'val': split_group(usable[n_train:n_train + n_val], rng),
        'test': split_group(usable[n_train + n_val:], rng),
    }


if CFG.SPLIT_PATH.exists():
    split = json.loads(CFG.SPLIT_PATH.read_text())
    print(f'Loaded existing split: {CFG.SPLIT_PATH}')
else:
    split = make_split()
    CFG.SPLIT_PATH.write_text(json.dumps(split, indent=1))
    print(f'Created split: {CFG.SPLIT_PATH}')

train_animals = split['train']
assert not set(train_animals) & set(split['val']['registered'] + split['val']['unknown'])
assert not set(train_animals) & set(split['test']['registered'] + split['test']['unknown'])

summary = pd.DataFrame([
    {'group': 'train', 'animals': len(train_animals),
     'images': sum(len(animals[a]) for a in train_animals)},
    *[{'group': g, 'animals': len(split[g]['registered']) + len(split[g]['unknown']),
       'registered': len(split[g]['registered']), 'unknown': len(split[g]['unknown']),
       'gallery images': sum(len(v) for v in split[g]['gallery'].values()),
       'registered queries': sum(q['registered'] for q in split[g]['queries']),
       'unknown queries': sum(not q['registered'] for q in split[g]['queries'])}
      for g in ('val', 'test')],
]).fillna('-')
summary

---

# Evaluation Protocol <a name="5"></a>

---

Every method produces a score matrix with one row per query and one column per gallery animal, where a higher score means more similar. The same functions then rank candidates, choose the threshold on validation unknown animals, and compute the metrics, so the methods differ only in how they score a pair of images.

In [ ]:
def top_candidates(scores: np.ndarray, gallery_animals: list[str]) -> tuple[np.ndarray, np.ndarray]:
    order = np.argsort(-scores, axis=1)
    return np.array(gallery_animals)[order], np.take_along_axis(scores, order, axis=1)


def pick_threshold(scores: np.ndarray, group: dict) -> float:
    unknown = np.array([not q['registered'] for q in group['queries']])
    top_unknown = scores[unknown].max(axis=1)
    return float(np.quantile(top_unknown, 1 - CFG.TARGET_UNKNOWN_FMR))


def compute_metrics(scores: np.ndarray, group: dict, threshold: float) -> dict:
    gallery_animals = list(group['gallery'])
    ranked, ranked_scores = top_candidates(scores, gallery_animals)
    truth = np.array([q['animal'] for q in group['queries']])
    registered = np.array([q['registered'] for q in group['queries']])
    top1, top1_score = ranked[:, 0], ranked_scores[:, 0]
    accepted = top1_score > threshold
    reg, unk = registered, ~registered
    return {
        'rank1': (top1[reg] == truth[reg]).mean(),
        'top5': np.mean([t in r[:5] for t, r in zip(truth[reg], ranked[reg])]),
        'correct_match': ((top1[reg] == truth[reg]) & accepted[reg]).mean(),
        'wrong_match_registered': ((top1[reg] != truth[reg]) & accepted[reg]).mean(),
        'unknown_fmr': accepted[unk].mean(),
        'threshold': threshold,
    }


results = {}
score_store = {}


def record(method: str, val_scores: np.ndarray, test_scores: np.ndarray) -> pd.DataFrame:
    threshold = pick_threshold(val_scores, split['val'])
    results[method] = {
        'val': compute_metrics(val_scores, split['val'], threshold),
        'test': compute_metrics(test_scores, split['test'], threshold),
    }
    score_store[method] = {'val': val_scores, 'test': test_scores}
    return pd.DataFrame(results[method]).T.round(4)

---

# Baseline: SIFT and RANSAC <a name="6"></a>

---

The classical baseline needs no training. SIFT keypoints are extracted after contrast equalisation, a FLANN index over all gallery descriptors shortlists the most-voted gallery images, and RANSAC keeps only matches that agree on one geometric transform. The score is the number of RANSAC inliers, which a verifier can inspect directly.

In [ ]:
def sift_features(rel: str):
    img = cv2.imread(str(CFG.IMAGE_ROOT / rel), cv2.IMREAD_GRAYSCALE)
    scale = CFG.SIFT_MAX_SIDE / max(img.shape)
    if scale < 1:
        img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    img = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)
    kps, desc = cv2.SIFT_create(nfeatures=CFG.SIFT_FEATURES).detectAndCompute(img, None)
    if desc is None or len(kps) < CFG.SIFT_MIN_MATCH:
        return None
    return np.float32([k.pt for k in kps]), desc


def ransac_inliers(a, b) -> int:
    pairs = cv2.BFMatcher(cv2.NORM_L2).knnMatch(a[1], b[1], k=2)
    good = [m for m, n in (p for p in pairs if len(p) == 2) if m.distance < CFG.SIFT_RATIO * n.distance]
    if len(good) < CFG.SIFT_MIN_MATCH:
        return 0
    src = a[0][[m.queryIdx for m in good]]
    dst = b[0][[m.trainIdx for m in good]]
    _, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    return 0 if mask is None else int(mask.sum())


def sift_scores(group: dict, feats: dict) -> np.ndarray:
    gallery_animals = list(group['gallery'])
    gallery = [(ai, img) for ai, a in enumerate(gallery_animals) for img in group['gallery'][a] if feats[img]]
    index = cv2.FlannBasedMatcher(dict(algorithm=1, trees=4), dict(checks=64))
    for _, img in gallery:
        index.add([feats[img][1]])
    index.train()

    def score_query(q: dict) -> np.ndarray:
        row = np.zeros(len(gallery_animals))
        f = feats[q['image']]
        if f is None:
            return row
        votes = Counter(m[0].imgIdx for m in index.knnMatch(f[1], k=1) if m)
        for gi, _ in votes.most_common(CFG.SIFT_SHORTLIST):
            ai, img = gallery[gi]
            row[ai] = max(row[ai], ransac_inliers(f, feats[img]))
        return row

    return np.stack([score_query(q) for q in tqdm(group['queries'], desc='SIFT queries')])

Features are extracted once for every validation and test image in parallel threads, because OpenCV releases the Python interpreter lock during extraction.

In [ ]:
eval_images = sorted({
    img for g in ('val', 'test')
    for img in [i for v in split[g]['gallery'].values() for i in v] + [q['image'] for q in split[g]['queries']]
})

t0 = time.time()
with ThreadPoolExecutor() as ex:
    sift_feats = dict(zip(eval_images, ex.map(sift_features, eval_images)))
print(f'SIFT features for {len(eval_images)} images in {time.time() - t0:.0f}s, '
      f'{sum(v is None for v in sift_feats.values())} with too few keypoints')

sift_val = sift_scores(split['val'], sift_feats)
sift_test = sift_scores(split['test'], sift_feats)
record('sift_ransac', sift_val, sift_test)

---

# Zero-Shot Foundation Models <a name="7"></a>

---

Before any fine-tuning, DINOv2 and DINOv3 are used exactly as released. Each image becomes the backbone's pooled embedding, each registered animal is represented by the mean of its gallery embeddings, and the score is cosine similarity. This measures how much muzzle identity the pretrained features already carry and decides which backbone to fine-tune.

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, items: list[str], labels: list[int] | None, transform):
        self.items, self.labels, self.transform = items, labels, transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        img = self.transform(Image.open(CFG.IMAGE_ROOT / self.items[i]).convert('RGB'))
        return img, (self.labels[i] if self.labels is not None else -1)


def create_backbone(name: str) -> nn.Module:
    return timm.create_model(name, pretrained=True, num_classes=0, img_size=CFG.IMG_SIZE)


def eval_transform(backbone: nn.Module):
    cfg = timm.data.resolve_data_config({}, model=backbone)
    return T.Compose([
        T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(cfg['mean'], cfg['std']),
    ])


@torch.no_grad()
def embed(model: nn.Module, images: list[str], transform) -> dict[str, np.ndarray]:
    model.eval()
    loader = DataLoader(ImageDataset(images, None, transform), batch_size=64,
                        num_workers=CFG.NUM_WORKERS, shuffle=False)
    out = []
    for x, _ in tqdm(loader, desc='embed', leave=False):
        out.append(F.normalize(model(x.to(CFG.DEVICE)).float(), dim=1).cpu().numpy())
    return dict(zip(images, np.concatenate(out)))


def cosine_scores(group: dict, emb: dict[str, np.ndarray]) -> np.ndarray:
    gallery = np.stack([np.mean([emb[i] for i in imgs], axis=0) for imgs in group['gallery'].values()])
    gallery /= np.linalg.norm(gallery, axis=1, keepdims=True)
    queries = np.stack([emb[q['image']] for q in group['queries']])
    return queries @ gallery.T

Both backbones are scored on the same validation and test animals as the SIFT baseline.

In [ ]:
for name in CFG.ZERO_SHOT_BACKBONES:
    backbone = create_backbone(name).to(CFG.DEVICE)
    emb = embed(backbone, eval_images, eval_transform(backbone))
    method = 'zeroshot_' + name.split('.')[0]
    display(record(method, cosine_scores(split['val'], emb), cosine_scores(split['test'], emb)))
    del backbone

pd.DataFrame({m: r['val'] for m, r in results.items()}).T.round(4)

---

# DINOv3 and ArcFace Fine-Tuning <a name="8"></a>

---

Fine-tuning uses only the training animals. The backbone's early blocks stay frozen to keep the general features learned during pretraining, and only the last blocks, a projection layer, and the ArcFace head are trained. After every epoch the model is evaluated on validation animals with the same open-set protocol, and the checkpoint with the best correct-match rate is kept.

## Training Data and Augmentation

Augmentation imitates frames from a handheld phone video: small changes of crop, rotation, light, and focus. Horizontal flips are excluded, because a mirrored muzzle is a different ridge pattern rather than the same animal.

In [ ]:
train_items = [img for a in train_animals for img in animals[a]]
label_of = {a: i for i, a in enumerate(train_animals)}
train_labels = [label_of[a] for a in train_animals for _ in animals[a]]

data_cfg = timm.data.resolve_data_config({}, model=create_backbone(CFG.BACKBONE))
train_transform = T.Compose([
    T.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.7, 1.0), ratio=(0.85, 1.18)),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.3),
    T.ToTensor(),
    T.Normalize(data_cfg['mean'], data_cfg['std']),
])
test_transform = T.Compose([
    T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(data_cfg['mean'], data_cfg['std']),
])

train_loader = DataLoader(
    ImageDataset(train_items, train_labels, train_transform),
    batch_size=CFG.BATCH_SIZE, shuffle=True, drop_last=True, num_workers=CFG.NUM_WORKERS,
)
print(f'Training animals: {len(train_animals)}  images: {len(train_items)}  batches/epoch: {len(train_loader)}')

## Model

The embedding model maps an image to a unit-length 512-dimensional vector. ArcFace compares that vector with one learned direction per training animal and adds an angular margin to the correct animal's angle, so the model must separate identities by more than the margin.

In [ ]:
class MuzzleEmbedder(nn.Module):
    def __init__(self, backbone_name: str):
        super().__init__()
        self.backbone = create_backbone(backbone_name)
        self.proj = nn.Sequential(
            nn.Linear(self.backbone.num_features, CFG.EMBED_DIM),
            nn.BatchNorm1d(CFG.EMBED_DIM),
        )

    def forward(self, x):
        return F.normalize(self.proj(self.backbone(x)), dim=1)


class ArcFaceHead(nn.Module):
    def __init__(self, n_classes: int):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_classes, CFG.EMBED_DIM))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m, self.sin_m = math.cos(CFG.ARC_MARGIN), math.sin(CFG.ARC_MARGIN)
        self.th = math.cos(math.pi - CFG.ARC_MARGIN)
        self.mm = math.sin(math.pi - CFG.ARC_MARGIN) * CFG.ARC_MARGIN

    def forward(self, emb, labels):
        cos = emb @ F.normalize(self.weight, dim=1).T
        sin = torch.sqrt((1.0 - cos ** 2).clamp(1e-7, 1.0))
        # cos(theta + m) = cos(theta)cos(m) - sin(theta)sin(m)
        cos_margin = cos * self.cos_m - sin * self.sin_m
        cos_margin = torch.where(cos > self.th, cos_margin, cos - self.mm)
        one_hot = F.one_hot(labels, cos.shape[1]).bool()
        return torch.where(one_hot, cos_margin, cos) * CFG.ARC_SCALE


def freeze_early_blocks(model: MuzzleEmbedder) -> None:
    for p in model.backbone.parameters():
        p.requires_grad = False
    for block in model.backbone.blocks[-CFG.TRAINABLE_BLOCKS:]:
        for p in block.parameters():
            p.requires_grad = True
    for p in model.backbone.norm.parameters():
        p.requires_grad = True


model = MuzzleEmbedder(CFG.BACKBONE).to(CFG.DEVICE)
head = ArcFaceHead(len(train_animals)).to(CFG.DEVICE)
freeze_early_blocks(model)
train_model = nn.DataParallel(model) if CFG.N_GPUS > 1 else model
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable backbone + projection parameters: {n_trainable / 1e6:.1f}M  GPUs used: {max(1, CFG.N_GPUS)}')

## Training Loop

The backbone and the new layers use separate learning rates, with a linear warmup followed by cosine decay. On Kaggle's T4 x2 machine the batch is split across both GPUs with `DataParallel`, and mixed precision is used on CUDA only. Validation after each epoch reuses the evaluation protocol from Section 5, so the selected checkpoint is optimised for open-set identification rather than training accuracy.

In [ ]:
def validate(model: nn.Module) -> dict:
    group = split['val']
    images = [i for v in group['gallery'].values() for i in v] + [q['image'] for q in group['queries']]
    scores = cosine_scores(group, embed(model, images, test_transform))
    return compute_metrics(scores, group, pick_threshold(scores, group))


def train() -> pd.DataFrame:
    optimizer = torch.optim.AdamW([
        {'params': [p for p in model.backbone.parameters() if p.requires_grad], 'lr': CFG.LR_BACKBONE},
        {'params': list(model.proj.parameters()) + list(head.parameters()), 'lr': CFG.LR_HEAD},
    ], weight_decay=CFG.WEIGHT_DECAY)
    total = CFG.EPOCHS * len(train_loader)
    warmup = CFG.WARMUP_EPOCHS * len(train_loader)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda s: (s + 1) / warmup if s < warmup else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total - warmup))),
    )
    use_amp = CFG.DEVICE == 'cuda'
    scaler = torch.amp.GradScaler(enabled=use_amp)

    history, best = [], -1.0
    for epoch in range(1, CFG.EPOCHS + 1):
        train_model.train()
        head.train()
        losses, t0 = [], time.time()
        for x, y in tqdm(train_loader, desc=f'epoch {epoch}', leave=False):
            x, y = x.to(CFG.DEVICE), y.to(CFG.DEVICE)
            with torch.autocast('cuda', enabled=use_amp):
                loss = F.cross_entropy(head(train_model(x), y), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            losses.append(loss.item())
        m = validate(model)
        history.append({'epoch': epoch, 'loss': np.mean(losses), **m, 'seconds': time.time() - t0})
        flag = ''
        if m['correct_match'] > best:
            best = m['correct_match']
            torch.save(model.state_dict(), CFG.RUN_DIR / 'best.pt')
            flag = '  <- best'
        print(f"epoch {epoch:2d}  loss {np.mean(losses):.3f}  val rank1 {m['rank1']:.3f}  "
              f"correct {m['correct_match']:.3f}  unk_fmr {m['unknown_fmr']:.3f}{flag}")
    history = pd.DataFrame(history)
    history.to_csv(CFG.RUN_DIR / 'history.csv', index=False)
    return history


if CFG.RUN_NAME == 'INFERENCE':
    history = None
else:
    history = train()

The loss and the validation metrics are plotted per epoch to check that open-set performance improves together with the training loss rather than overfitting the training animals.

In [ ]:
if history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    axes[0].plot(history['epoch'], history['loss'], marker='o')
    axes[0].set(title='ArcFace training loss', xlabel='Epoch', ylabel='Loss')
    for col in ('rank1', 'correct_match', 'unknown_fmr'):
        axes[1].plot(history['epoch'], history[col], marker='o', label=col)
    axes[1].set(title='Validation (open set)', xlabel='Epoch', ylim=(0, 1))
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(CFG.FIG_DIR / 'training.png', dpi=120)
    plt.show()

The best checkpoint is reloaded and scored once on validation and test animals with the shared protocol. The test animals were not used to choose the checkpoint or the threshold.

In [ ]:
model.load_state_dict(torch.load(CFG.RUN_DIR / 'best.pt', map_location=CFG.DEVICE))
ft_emb = embed(model, eval_images, test_transform)
record('dinov3_arcface', cosine_scores(split['val'], ft_emb), cosine_scores(split['test'], ft_emb))

---

# Hybrid: Embedding Retrieval and SIFT Verification <a name="9"></a>

---

Cosine similarity ranks candidates well, but a cosine threshold chosen on validation animals does not transfer reliably: its meaning shifts with gallery size and with the few unknown animals available for calibration. A RANSAC inlier count measures geometric agreement between two specific muzzle patterns, so its threshold keeps the same meaning across galleries. The hybrid uses the fine-tuned embedding to shortlist the top candidates and SIFT with RANSAC to verify them.

In [ ]:
def hybrid_scores(group: dict, emb: dict[str, np.ndarray]) -> np.ndarray:
    gallery_animals = list(group['gallery'])
    cos = cosine_scores(group, emb)
    shortlist = np.argsort(-cos, axis=1)[:, :CFG.HYBRID_TOP_K]

    def verify(qi: int) -> np.ndarray:
        row = np.zeros(len(gallery_animals))
        f = sift_feats[group['queries'][qi]['image']]
        if f is None:
            return row
        for ai in shortlist[qi]:
            imgs = group['gallery'][gallery_animals[ai]]
            row[ai] = max(ransac_inliers(f, sift_feats[i]) for i in imgs if sift_feats[i] is not None)
        return row

    with ThreadPoolExecutor() as ex:
        return np.stack(list(tqdm(ex.map(verify, range(len(group['queries']))),
                                  total=len(group['queries']), desc='verify')))


record('hybrid_dinov3_sift', hybrid_scores(split['val'], ft_emb), hybrid_scores(split['test'], ft_emb))

---

# Results Comparison <a name="10"></a>

---

All methods are compared on the test animals with thresholds chosen on validation animals. A method qualifies only if its validation threshold still holds on test, meaning the unknown-animal false-match rate stays within twice the 5% target. Qualifying methods are then ranked by `correct_match`.

In [ ]:
leaderboard = pd.DataFrame({m: r['test'] for m, r in results.items()}).T
leaderboard['threshold_holds'] = leaderboard['unknown_fmr'] <= 2 * CFG.TARGET_UNKNOWN_FMR
leaderboard = leaderboard.sort_values(['threshold_holds', 'correct_match'], ascending=False).round(4)
best_method = leaderboard.index[0]

(CFG.RESULT_DIR / 'muzzle_leaderboard.json').write_text(json.dumps(
    {m: {s: {k: float(v) for k, v in r[s].items()} for s in r} for m, r in results.items()}, indent=1))
(CFG.RESULT_DIR / 'muzzle_leaderboard.md').write_text(
    f'# Muzzle leaderboard (test animals, split {CFG.SPLIT_PATH.name})\n\n' + leaderboard.to_markdown() + '\n')
print(f'Best method on test: {best_method}')
leaderboard

The score distributions of the best method show how far apart correct matches and unknown animals sit. A wide gap means the threshold is robust; overlap means more cases will be sent to the verifier.

In [ ]:
group = split['test']
scores = score_store[best_method]['test']
ranked, ranked_scores = top_candidates(scores, list(group['gallery']))
truth = np.array([q['animal'] for q in group['queries']])
reg = np.array([q['registered'] for q in group['queries']])
correct = reg & (ranked[:, 0] == truth)
wrong = reg & (ranked[:, 0] != truth)
threshold = results[best_method]['test']['threshold']

plt.figure(figsize=(9, 3.5))
bins = np.linspace(ranked_scores[:, 0].min(), ranked_scores[:, 0].max(), 50)
plt.hist(ranked_scores[correct, 0], bins=bins, alpha=0.6, label='registered, correct top-1')
plt.hist(ranked_scores[wrong, 0], bins=bins, alpha=0.6, label='registered, wrong top-1')
plt.hist(ranked_scores[~reg, 0], bins=bins, alpha=0.6, label='unknown animal')
plt.axvline(threshold, color='black', linestyle='--', label='threshold (from val)')
plt.title(f'Top-1 score distribution on test animals: {best_method}')
plt.xlabel('Score')
plt.ylabel('Queries')
plt.legend()
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'score_distribution.png', dpi=120)
plt.show()

---

# Export to ONNX <a name="11"></a>

---

The fine-tuned embedder is exported to ONNX so the Veyra vision service can run it with `onnxruntime` and no PyTorch dependency. The ArcFace head is not exported, because inference only needs embeddings. The exported model is checked against the PyTorch model on real images, and a model card records the data, split, metrics, and limits.

In [ ]:
export_model = MuzzleEmbedder(CFG.BACKBONE)
export_model.load_state_dict(torch.load(CFG.RUN_DIR / 'best.pt', map_location='cpu'))
export_model.eval()

onnx_path = CFG.EXPORT_DIR / 'muzzle_embed_v1.onnx'
dummy = torch.randn(1, 3, CFG.IMG_SIZE, CFG.IMG_SIZE)
torch.onnx.export(
    export_model, dummy, onnx_path,
    input_names=['image'], output_names=['embedding'],
    dynamic_axes={'image': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17, dynamo=False,
)

check_images = eval_images[:16]
batch = torch.stack([test_transform(Image.open(CFG.IMAGE_ROOT / i).convert('RGB')) for i in check_images])
with torch.no_grad():
    torch_out = export_model(batch).numpy()
onnx_out = ort.InferenceSession(str(onnx_path)).run(None, {'image': batch.numpy()})[0]
print(f'ONNX file: {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} MB)')
print(f'Max abs difference vs PyTorch: {np.abs(torch_out - onnx_out).max():.2e}')

The model card is written next to the ONNX file so that anyone deploying the model sees its evidence and its limits together.

In [ ]:
r = results['hybrid_dinov3_sift']
card = f'''# muzzle_embed_v1

- Task: open-set individual cattle identification from muzzle crops (retrieval, not classification).
- Backbone: {CFG.BACKBONE} (DINOv3 License), last {CFG.TRAINABLE_BLOCKS} blocks fine-tuned with ArcFace.
- Input: RGB {CFG.IMG_SIZE}x{CFG.IMG_SIZE}, normalised with mean {data_cfg['mean']} and std {data_cfg['std']}.
- Output: L2-normalised {CFG.EMBED_DIM}-d embedding used to shortlist the top {CFG.HYBRID_TOP_K} gallery animals by cosine similarity.
- Verification: SIFT + ratio test + RANSAC on the shortlist (max side {CFG.SIFT_MAX_SIDE}px, CLAHE); score = inlier count.
- Accept a match when inliers > {r['test']['threshold']:.0f} (chosen on validation animals for unknown false-match rate <= {CFG.TARGET_UNKNOWN_FMR:.0%}). Cosine alone is not used for acceptance: its validation threshold did not transfer to test.
- Data: Beef Cattle Muzzle database, Li et al. 2022, Zenodo 6324361, CC BY 4.0. Split {CFG.SPLIT_PATH.name}, by animal.

## Test animals, hybrid pipeline (never used for training, checkpoint, or threshold)

| Metric | Value |
| --- | --- |
| Correct match at threshold | {r['test']['correct_match']:.3f} |
| Unknown-animal false-match rate | {r['test']['unknown_fmr']:.3f} |
| Rank-1 | {r['test']['rank1']:.3f} |
| Top-5 | {r['test']['top5']:.3f} |

## Limits

- US feedlot cattle and cropped still photographs; not Chinese breeds and not phone video frames.
- Images come from one collection period; identification across months is untested.
- Scores are similarities, not calibrated probabilities. A match is a candidate for human verification.
'''
(CFG.EXPORT_DIR / 'MODEL_CARD.md').write_text(card)
print(card)